In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Install the latest transformers framework = Done
#!pip install -q transformers accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
print("Libraries imported successfully!")
import os
from transformers import AutoTokenizer, AutoModelForCausalLM


/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model.safetensors.index.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model-00001-of-00002.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/model-00002-of-00002.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/README.md
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer_config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/special_tokens_map.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/.gitattributes
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/tokenizer.model
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2/generation_config.json
Libraries imported successfully!


In [13]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("🔍 Scanning your Kaggle workspace for Gemma 2...")

# This loop automatically finds where Kaggle hid the configuration file
model_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files:
        model_dir = root
        break

if model_dir:
    print(f"✅ Found the model! Exact path is: {model_dir}")
    
    # Load the tokenizer using the auto-discovered path
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)

    # Load the model layers onto the GPU
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=True
    )
    print("\n🚀 SUCCESS! Gemma 2 is fully loaded onto your GPU and ready to think!")
else:
    print("❌ Could not find Gemma 2 in your sidebar. Please look at the top right of your sidebar under 'Input' and verify that 'gemma-2' is listed there.")

🔍 Scanning your Kaggle workspace for Gemma 2...
✅ Found the model! Exact path is: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]


🚀 SUCCESS! Gemma 2 is fully loaded onto your GPU and ready to think!


In [18]:
# Create your prompt
prompt_text = """
You are an expert change management consultant specializing in helping small retail businesses.
A local boutique clothing store wants to move from manual excel tracking to an automated inventory system.

Provide 3 quick, practical, low-risk steps they should take this week to prepare their data and staff for this transition.
"""

# Format the text into Gemma's chat style
messages = [{"role": "user", "content": prompt_text}]

# FIX 1: Add return_dict=True and explicitly grab the 'input_ids' tensor key
encoded_inputs = tokenizer.apply_chat_template(
    messages, 
    return_tensors="pt", 
    add_generation_prompt=True,
    return_dict=True
)

# Move the text tokens onto the GPU
input_ids = encoded_inputs["input_ids"].to("cuda")

# Tell Gemma to generate the text
with torch.no_grad():
    outputs = model.generate(
        input_ids, 
        max_new_tokens=400, 
        temperature=0.7,  
        do_sample=True
    )

# FIX 2: Slice the output using the exact length of the input_ids array
input_length = input_ids.shape[1]
generated_tokens = outputs[0][input_length:]

# Decode and print the clean result
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(response)

Here are 3 quick, practical, and low-risk steps a small retail boutique can take this week to prepare for transitioning to an automated inventory system:

**1. Data Audit & Categorization (Day 1 & 2):**

* **What to do:**  Work with the store owner/manager to collect and review all current inventory data.  This includes: 
    * SKU (Stock Keeping Unit) numbers for each item
    * Item descriptions
    * Current quantity on hand
    * Purchase date and cost
    * Supplier information 
    *  If applicable, any historical sales data.
* **Why it's beneficial:**  A clear picture of the current inventory situation is essential. This data will be the foundation for the new system and allows the team to identify any inconsistencies or missing information. 

**2. Staff Training (Day 3):**

* **What to do:**  Organize a short training session for the staff (around 1 hour) to introduce the new inventory system and its features. 
    * Focus on: 
        * How to access the system
        * Basic